## OPTIMAL EVENT TRADING

Optimal execution around scheduled events (e.g. earnings, macro announcements): modeling optimal timing and sizing of trades around a known event time.

In [51]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [52]:
# Standard library imports

# Third-party imports
import numpy as np
import plotly.graph_objects as go
import polars as pl

# First-party imports
from xpectral.quant.execution import (
    cost_variance,
    decay_parameter,
    efficient_frontier,
    expected_cost,
    optimal_holdings,
    permanent_impact_cost,
    temporary_impact_cost,
)

## Optimal holdings trajectory

The Almgren-Chriss closed-form schedule, $x(t) = X \cdot \sinh(\kappa(T-t)) / \sinh(\kappa T)$,
for the three illustrative cases from `almgren-chriss-explained.md`: risk-neutral
($\lambda=0$, a straight line), moderate ($\kappa=2$), and aggressive ($\kappa=5$).

In [53]:
X = 1.0
T = 1.0
t_grid = np.linspace(0.0, T, 101)

kappas = {"risk_neutral": 0.0, "moderate": 2.0, "aggressive": 5.0}
colors = {"risk_neutral": "#1f77b4", "moderate": "#ff7f0e", "aggressive": "#2ca02c"}

trajectories_df = pl.DataFrame(
    {
        "t_fraction": t_grid / T,
        **{
            label: optimal_holdings(t_grid, X, T, kappa)
            for label, kappa in kappas.items()
        },
    }
)

fig = go.Figure()
for label, kappa in kappas.items():
    legend_suffix = "λ=0" if kappa == 0.0 else f"κ={kappa:g}"
    fig.add_trace(
        go.Scatter(
            x=trajectories_df["t_fraction"],
            y=trajectories_df[label],
            mode="lines",
            name=f"holdings ({legend_suffix})",
            line={"width": 2, "color": colors[label]},
        )
    )
fig.update_layout(
    title="Optimal holdings trajectory x(t)/X",
    xaxis_title="Fraction of time elapsed",
    yaxis_title="Fraction of shares held",
    width=700,
    height=400,
    legend={"x": 1, "xanchor": "right", "y": 0.5, "yanchor": "middle"},
)
fig.show()

## Efficient frontier

Sweeping the risk-aversion parameter $\lambda$ traces out the efficient frontier:
the lowest achievable expected cost for each level of cost variance (risk). Expected
cost splits into a permanent-impact piece, $\frac{1}{2}\gamma X^2$, fixed by the total
size $X$ and identical for every schedule, and a temporary-impact piece that shrinks as
the schedule spreads trading over more time (lower risk aversion, lower $\kappa$).

Stock price enters here through $\sigma$: dollar volatility per share is the stock's
*price* times its *percentage* volatility, so the same percentage-vol stock at a higher
price has a wider, riskier frontier for the same share count $X$.

In [54]:
# Illustrative liquidation of a position in a $50 stock over one trading day.
price = 50.0  # $/share
X = 1_000.0  # shares
T = 1.0
sigma_pct = 0.01  # daily volatility, as a fraction of price
sigma = sigma_pct * price  # $/share/sqrt(day)
eta = 2.5e-6  # temporary impact coefficient
gamma = 2.5e-7  # permanent impact coefficient

print(f"notional: ${X * price:,.0f}, sigma: ${sigma:.2f}/share/sqrt(day)")

lambdas = np.concatenate([[0.0], np.logspace(-10, -2, 49)])

frontier = efficient_frontier(lambdas, X, T, sigma, eta, gamma)
frontier_df = pl.DataFrame(
    {
        "variance": frontier["variance"],
        "expected_cost": frontier["expected_cost"],
        "permanent_cost": permanent_impact_cost(X, gamma),
        "temporary_cost": [
            temporary_impact_cost(X, eta, T, kappa) for kappa in frontier["kappa"]
        ],
    }
).sort("variance")
frontier_df = frontier_df.with_columns(cost_std=frontier_df["variance"].sqrt())

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=frontier_df["cost_std"],
        y=frontier_df["expected_cost"],
        mode="lines+markers",
        name="expected cost",
        line={"width": 2},
        marker={"size": 5},
    )
)
fig.add_trace(
    go.Scatter(
        x=frontier_df["cost_std"],
        y=frontier_df["permanent_cost"],
        mode="lines",
        name="permanent-impact cost",
        line={"width": 2, "dash": "dash"},
    )
)
fig.add_trace(
    go.Scatter(
        x=frontier_df["cost_std"],
        y=frontier_df["temporary_cost"],
        mode="lines",
        name="temporary-impact cost",
        line={"width": 2, "dash": "dot"},
    )
)
fig.update_layout(
    title="Almgren-Chriss efficient frontier and cost composition",
    xaxis_title="Std. deviation of cost ($)",
    yaxis_title="Cost ($)",
    width=700,
    height=400,
    legend={"x": 1, "xanchor": "right", "y": 1, "yanchor": "top"},
)
fig.show()

notional: $50,000, sigma: $0.50/share/sqrt(day)


## Naive-strategy benchmarking

The Almgren-Chriss trajectory is derived by minimizing a mean-variance objective,
but most real trading schedules don't solve that optimization; they just follow
a shape. Here we benchmark the AC efficient frontier against a family of "naive"
power-law schedules $x(t) = X \cdot (1 - t/T)^p$: $p=1$ is TWAP, $p>1$ front-loads
(dumps early), $p<1$ back-loads (procrastinates, then dumps late). None of these
are AC-optimal for any $\lambda$, so their (variance, cost) pairs should sit
dominated, above/right of the efficient frontier traced out earlier.

In [55]:
def power_law_holdings(t: np.ndarray, X: float, T: float, p: float) -> np.ndarray:
    return X * (1.0 - t / T) ** p


def schedule_cost_variance(
    t: np.ndarray,
    x: np.ndarray,
    X: float,
    sigma: float,
    eta: float,
    gamma: float,
) -> tuple[float, float]:
    """Cost and variance of an arbitrary (not necessarily AC-optimal) schedule."""
    trading_rate = -np.gradient(x, t)
    cost = permanent_impact_cost(X, gamma) + eta * np.trapezoid(trading_rate**2, t)
    variance = sigma**2 * np.trapezoid(x**2, t)
    return cost, variance


t_fine = np.linspace(0.0, T, 2001)

# Sanity check: numeric integration should reproduce the closed-form AC
# formulas for a trajectory that actually is AC-optimal.
kappa_check = decay_parameter(1e-6, sigma, eta)
x_check = optimal_holdings(t_fine, X, T, kappa_check)
numeric_cost, numeric_variance = schedule_cost_variance(
    t_fine, x_check, X, sigma, eta, gamma
)
print(
    f"numeric cost {numeric_cost:.6e} vs closed-form "
    f"{expected_cost(X, gamma, eta, T, kappa_check):.6e}"
)
print(
    f"numeric variance {numeric_variance:.6e} vs closed-form "
    f"{cost_variance(X, sigma, T, kappa_check):.6e}"
)

numeric cost 2.625545e+00 vs closed-form 2.625545e+00
numeric variance 8.223790e+04 vs closed-form 8.223789e+04


In [56]:
naive_ps = {
    "back-loaded (p=0.25)": 0.25,
    "TWAP (p=1)": 1.0,
    "front-loaded (p=2)": 2.0,
    "aggressive dump (p=6)": 6.0,
}

fig = go.Figure()
for label, p in naive_ps.items():
    fig.add_trace(
        go.Scatter(
            x=t_fine / T,
            y=power_law_holdings(t_fine, X, T, p) / X,
            mode="lines",
            name=label,
            line={"width": 2},
        )
    )
fig.update_layout(
    title="Naive power-law liquidation schedules x(t)/X",
    xaxis_title="Fraction of time elapsed",
    yaxis_title="Fraction of shares held",
    width=700,
    height=400,
)
fig.show()

In [57]:
naive_results = []
for label, p in naive_ps.items():
    x_p = power_law_holdings(t_fine, X, T, p)
    cost_p, variance_p = schedule_cost_variance(t_fine, x_p, X, sigma, eta, gamma)
    naive_results.append({"strategy": label, "cost": cost_p, "variance": variance_p})
naive_df = pl.DataFrame(naive_results)
naive_df = naive_df.with_columns(cost_std=naive_df["variance"].sqrt())

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=frontier_df["cost_std"],
        y=frontier_df["expected_cost"],
        mode="lines",
        name="AC efficient frontier",
        line={"width": 2},
    )
)
fig.add_trace(
    go.Scatter(
        x=naive_df["cost_std"],
        y=naive_df["cost"],
        mode="markers+text",
        name="naive strategies",
        text=naive_df["strategy"],
        textposition="top center",
        marker={"size": 10, "symbol": "diamond", "color": "#d62728"},
    )
)
fig.update_layout(
    title="Naive strategies vs. the Almgren-Chriss efficient frontier (log-log)",
    xaxis={"title": "Std. deviation of cost ($)", "type": "log"},
    yaxis={"title": "Cost ($)", "type": "log"},
    width=800,
    height=450,
    legend={"x": 0.02, "y": 0.98},
)
fig.show()

## Monte Carlo validation

`expected_cost` and `cost_variance` come from solving the continuous-time
problem analytically. To check that algebra against an actual trading
simulation: discretize the optimal trajectory into $N$ trades, simulate $M$
random price paths under the model's own dynamics (a random walk shifted by
each trade's permanent impact, with temporary impact reducing the fill price
of that trade), and compare the empirical mean/variance of realized cost
across paths to the closed-form values.

In [58]:
# Reuses the liquidation scenario (X, T, sigma, eta, gamma) from the efficient
# frontier above; picks one point on that frontier to simulate.
lambda_mc = 1e-6
kappa_mc = decay_parameter(lambda_mc, sigma, eta)

N = 250  # discrete trades
M = 100_000  # simulated price paths
tau = T / N

t_grid_mc = np.linspace(0.0, T, N + 1)
x_mc = optimal_holdings(t_grid_mc, X, T, kappa_mc)
n = -np.diff(x_mc)  # shares sold in each interval, length N

# Deterministic components of realized cost: permanent impact accumulated
# against prior trades, and temporary impact on each trade's own fill price.
cumsum_before_n = np.concatenate(([0.0], np.cumsum(n)[:-1]))
permanent_cost_discrete = gamma * np.sum(n * cumsum_before_n)
temporary_cost_discrete = eta / tau * np.sum(n**2)

rng = np.random.default_rng(0)
xi = rng.standard_normal((M, N))
cumsum_before_xi = np.concatenate(
    [np.zeros((M, 1)), np.cumsum(xi, axis=1)[:, :-1]], axis=1
)
noise = -sigma * np.sqrt(tau) * (cumsum_before_xi @ n)

simulated_costs = permanent_cost_discrete + temporary_cost_discrete + noise

# At small position sizes, the expected cost can be small relative to its own
# variance, so a single point estimate of the mean is noisy: report its
# standard error and check the closed form against a 95% confidence interval
# rather than expecting an exact match.
mean_stderr = simulated_costs.std(ddof=1) / np.sqrt(M)
mc_variance = simulated_costs.var()
closed_form_variance = cost_variance(X, sigma, T, kappa_mc)

comparison_df = pl.DataFrame(
    {
        "quantity": ["mean cost", "cost variance", "cost std. deviation"],
        "monte_carlo": [simulated_costs.mean(), mc_variance, np.sqrt(mc_variance)],
        "monte_carlo_stderr": [mean_stderr, None, None],
        "closed_form": [
            expected_cost(X, gamma, eta, T, kappa_mc),
            closed_form_variance,
            np.sqrt(closed_form_variance),
        ],
    }
)
comparison_df

quantity,monte_carlo,monte_carlo_stderr,closed_form
str,f64,f64,f64
"""mean cost""",2.126363,0.90064,2.625545
"""cost variance""",81114.349289,null,82237.886238
"""cost std. deviation""",284.80581,null,286.771488


In [59]:
analytic_mean = expected_cost(X, gamma, eta, T, kappa_mc)
analytic_std = np.sqrt(cost_variance(X, sigma, T, kappa_mc))
cost_range = np.linspace(simulated_costs.min(), simulated_costs.max(), 400)
normal_pdf = np.exp(-0.5 * ((cost_range - analytic_mean) / analytic_std) ** 2) / (
    analytic_std * np.sqrt(2 * np.pi)
)

fig = go.Figure()
fig.add_trace(
    go.Histogram(
        x=simulated_costs,
        histnorm="probability density",
        name="simulated cost",
        marker={"color": "#1f77b4"},
        opacity=0.75,
    )
)
fig.add_trace(
    go.Scatter(
        x=cost_range,
        y=normal_pdf,
        mode="lines",
        name="closed-form N(mean, variance)",
        line={"color": "#d62728", "width": 2},
    )
)
fig.update_layout(
    title=f"Monte Carlo cost distribution vs. closed-form (N={N} trades, M={M:,} paths)",
    xaxis_title="Realized cost ($)",
    yaxis_title="Probability density",
    width=800,
    height=450,
    legend={"x": 0.02, "y": 0.98},
)
fig.show()